## Cleaning & Formatting German News Data for Inference
This notebook prepares the **German** half of the bidirectional inference dataset. It transforms the raw German news articles (previously converted to JSONL) into the specific input format required by the NLLB model for translation into Odia.

### Key Achievements:
* **Data Integration:** Loaded the `german_news.jsonl` file containing **167,280** raw German news records.
* **Task-Specific Formatting:**
  * **Content Merging:** Concatenated the `title` and `text` fields to ensure the model translates the full context of each article.
  * **Prefix Injection:** Applied the necessary task prompt `"translate German to Odia: "` to every record, signaling the model to perform translation rather than continuation or summarization.
* **Output Generation:** Saved the processed dataset to `german_news_inference.jsonl`, creating a clean, standardized corpus ready to be fed into the inference pipeline for large-scale translation.

### Workflow Context:
* **Input:** `german_news.jsonl` (Raw text)
* **Process:** Merge Title/Body $\rightarrow$ Add Prefix
* **Output:** `german_news_inference.jsonl` (Inference-ready)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import json
import re

In [ ]:
# ==========================================
# CONFIGURATION
# ==========================================
INPUT_FILE = "/content/drive/MyDrive/Research_Paper_Publication/test/data/new_data/german_news.jsonl"
OUTPUT_FILE = "/content/drive/MyDrive/Research_Paper_Publication/test/data/new_data/cleaned/german_news_inference.jsonl"

In [ ]:
PREFIX_DEU_TO_ORI = "translate German to Odia: "

In [ ]:
cleaned_data = []

In [ ]:
print(f"Processing {INPUT_FILE}...")

with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        entry = json.loads(line)

        # 1. Check if 'text' is not null
        if entry.get('text'):

            # 2. Prepare the input string with the prefix
            # We combine title and text for a complete context,
            # or you can just use entry['text'] if you prefer.
            source_text = f"{entry.get('title', '')}: {entry['text']}"

            # 3. Create the new object structure
            new_entry = {
                # This matches the INPUT_FIELD in your training code
                "input_text": PREFIX_DEU_TO_ORI + source_text,

                # We add an empty target_text because your dataloaders
                # might expect this key to exist, even if empty for inference.
                "target_text": "",

                # Keep metadata for reference
                "original_source": entry.get('source'),
                "original_url": entry.get('url', 'N/A')
            }

            cleaned_data.append(new_entry)

Processing /content/drive/MyDrive/Research_Paper_Publication/test/data/new_data/german_news.jsonl...


In [ ]:
# Save to the new JSONL file
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    for entry in cleaned_data:
        json.dump(entry, f, ensure_ascii=False)
        f.write('\n')

In [ ]:
print(f"Success! Filtered and formatted {len(cleaned_data)} records to '{OUTPUT_FILE}'.")
print("Sample entry:")
print(json.dumps(cleaned_data[0], indent=2, ensure_ascii=False))

Success! Filtered and formatted 167280 records to '/content/drive/MyDrive/Research_Paper_Publication/test/data/new_data/cleaned/german_news_inference.jsonl'.
Sample entry:
{
  "input_text": "translate German to Odia: Warum nicht mal die Russen Putins Impfstoff wollen: Im Kreml sollte es in diesen Tagen eigentlich genug Gründe für Aufruhr geben. Vor gerade einem Monat schien die Covid-Pandemie in Russland nahezu eingedämmt. Doch in weniger als vier Wochen hat sich die Zahl der Neuinfektionen auf knapp 11.600 pro Tag mehr als verdoppelt und liegt nun auf dem Niveau der Rekordwerte von Mai. Die zweite Welle ist in Russland angekommen.\n\nDoch geredet wird darüber überraschend wenig. Der russische Staat gibt sich bislang betont gelassen. Gesundheitsminister Michail Muraschko nennt die Situation „kontrollierbar“. Das klingt selbstbewusst. Dabei steht Russland vor einer schier unmöglichen Aufgabe.",
  "target_text": "",
  "original_source": "welt",
  "original_url": "N/A"
}
